In [1]:
import json
import pandas as pd
from tqdm import tqdm

In [2]:
import os

DATASET_PATH = "/kaggle/input/llm-training-in-multilingual-languages"

for file in os.listdir(DATASET_PATH):
    print(file)


english_dataset.jsonl
sinhala_dataset.jsonl
tamil_dataset.jsonl


In [3]:
import json
import pandas as pd

def load_jsonl_safe(filepath):
    records = []
    bad_lines = 0

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                bad_lines += 1
                continue

    print(f"{filepath} → loaded {len(records)} rows, skipped {bad_lines} bad lines")
    return pd.DataFrame(records)


In [4]:
df_en = load_jsonl_safe(f"{DATASET_PATH}/english_dataset.jsonl")
df_ta = load_jsonl_safe(f"{DATASET_PATH}/tamil_dataset.jsonl")
df_si = load_jsonl_safe(f"{DATASET_PATH}/sinhala_dataset.jsonl")

/kaggle/input/llm-training-in-multilingual-languages/english_dataset.jsonl → loaded 1000 rows, skipped 0 bad lines
/kaggle/input/llm-training-in-multilingual-languages/tamil_dataset.jsonl → loaded 998 rows, skipped 2 bad lines
/kaggle/input/llm-training-in-multilingual-languages/sinhala_dataset.jsonl → loaded 996 rows, skipped 4 bad lines


In [5]:
df_en["language"] = "en"
df_si["language"] = "si"
df_ta["language"] = "ta"

In [6]:
df = pd.concat([df_en, df_si, df_ta], ignore_index=True)
df.shape

(2994, 4)

In [7]:
df.columns

Index(['user_emotion', 'user_input', 'bot_response', 'language'], dtype='object')

In [8]:
df = df.dropna(subset=["user_emotion", "user_input", "bot_response"])
df.shape

(2994, 4)

In [9]:
df = df[df["user_input"].str.strip().str.len() > 5]
df = df[df["bot_response"].str.strip().str.len() > 5]
df.shape

(2994, 4)

In [10]:
def clean_text(text):
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = " ".join(text.split())
    return text.strip()

df["user_input"] = df["user_input"].apply(clean_text)
df["bot_response"] = df["bot_response"].apply(clean_text)
df["user_emotion"] = df["user_emotion"].str.lower().str.strip()


In [11]:
df["user_emotion"].value_counts()

user_emotion
angry        429
disgust      429
happy        429
surprised    428
sad          427
fearful      426
neutral      426
Name: count, dtype: int64

In [12]:
df["prompt"] = (
    "Emotion: " + df["user_emotion"] + "\n"
    "User: " + df["user_input"] + "\n"
    "Assistant:"
)

In [13]:
df["target"] = df["bot_response"]

In [14]:
df["prompt_len"] = df["prompt"].str.len()
df["prompt_len"].describe()

count    2994.000000
mean       78.587174
std        14.534103
min        45.000000
25%        68.000000
50%        77.000000
75%        87.000000
max       133.000000
Name: prompt_len, dtype: float64

In [15]:
df["prompt"] = df["prompt"].str.slice(0, 800)
df["target"] = df["target"].str.slice(0, 800)

In [16]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42
)


In [17]:
train_df[["prompt", "target", "language"]].to_json(
    "train.jsonl", orient="records", lines=True
)

val_df[["prompt", "target", "language"]].to_json(
    "val.jsonl", orient="records", lines=True
)

test_df[["prompt", "target", "language"]].to_json(
    "test.jsonl", orient="records", lines=True
)